In [9]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

In [10]:
# Load relevant columns including cosine similarity
df_bt = pd.read_csv("files/welsh_back_translation_french.csv",
                    usecols=["back_translated_welsh", "cefr_level", "cosine_similarity"]) \
         .rename(columns={"back_translated_welsh": "text"})

In [11]:
# Filter by cosine_similarity > 0.80, then drop the column
df_bt = df_bt[df_bt["cosine_similarity"] > 0.80].drop(columns=["cosine_similarity"])

In [13]:
# Add missing columns
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [14]:
df_bt

,text,cefr_level,title,lang,source_name,format,category,license
12,"Ac yn y bore: Helo, mae hi'n braf iawn heddiw.",A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
34,Ac: es i brecwast heddiw! O mi cwrdd wy ar wy:...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
56,Pam bydd fy nhad yn awr yn mynd i'r gwely. Pam...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
57,Dywed mam: Deres fy helpu i rannu'r gegin. Pam...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
69,Ac: mae prynhawn dda. Ar hyn yn ofnadwy! Ydy w...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
1355,Ydych eisiau mynd i Affrica?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1356,A hoffech chi fynd i America?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1358,A fydden nhw'n mynd i'r Almaen?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1363,Ni allwn newid y olwyn.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [15]:
# Count how many samples are labeled A1 and A2
df_bt["cefr_level"].value_counts()

cefr_level
A1    199
A2     93
Name: count, dtype: int64

In [16]:
# Load Welsh CEFR dataset from HuggingFace
ds_welsh = load_dataset("UniversalCEFR/learn_welsh_cy")["train"].to_pandas()

In [17]:
ds_welsh

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,..."
1,Uned 1 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ..."
2,Uned 1 - Sgwrs 3,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b..."
3,Uned 2 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D..."
4,Uned 2 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...
...,...,...,...,...,...,...,...,...
1367,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allech chi gyrraedd yn gynnar?
1368,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allet ti gyrraedd yn gynnar?
1369,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allai hi gyrraedd yn gynnar?
1370,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allen nhw gyrraedd yn gynnar?


In [18]:
df_merged = pd.concat([ds_welsh, df_bt], ignore_index=True)
df_merged = df_merged.drop_duplicates(subset="text", keep="first").sample(frac=1, random_state=42)

In [19]:
df_merged

,title,lang,source_name,format,category,cefr_level,license,text
136,Uned 2 - Ynganu 13,cy,mynediad-de-learnwelsh,sentence-level,reference,A1,public,Ble dych chi'n siopa?
1284,Uned 17 - Cymharu ansoddeiriau eto,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Fe yw'r gwaetha.
1120,Uned 10 - Gwnaf i,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Gwnaf i'r coffi.
1001,Uned 3 - Baswn i,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Faswn i ddim yn lico yfed coffi du.
811,Uned 8 - Siaradwch - Darllen,cy,sylfaen-de-learnwelsh,dialogue-level,reference,A2,public,A: Beth ddarllenoch chi ddiwetha?\nB: Darllena...
...,...,...,...,...,...,...,...,...
1133,Uned 10 - Wnei di?,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Wnei di helpu gyda'r gwaith?
1297,Uned 17 -na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Caerdydd yw'r ddinas fwya swnllyd.
863,Uned 17 - Cwrs Sbaeneg,cy,sylfaen-de-learnwelsh,document-level,reference,A2,public,Croeso i bawb... os dych chi'n siarad tipyn o'...
1484,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A1,CC-BY-SA,Roedd peswch mewn ohonynt.


In [20]:
df_merged["cefr_level"].value_counts()

cefr_level
A1    926
A2    688
Name: count, dtype: int64

In [21]:
hf_dataset=Dataset.from_pandas(df_merged.reset_index(drop=True))

In [22]:
hf_dataset

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 1614
})

In [23]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in hf_dataset["cefr_level"]])

In [24]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [25]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [26]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [27]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [28]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(hf_dataset, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = hf_dataset.select(train_idx)
    ds_val = hf_dataset.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_DA_fr/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 323/323 [00:00<00:00, 9629.60 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_29068\2799807945.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.893600,0.782814,0.575851,0.420858,0.331605,0.575851,0.575851,1.000000,0.730845,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.569200,0.377863,0.845201,0.843593,0.846334,0.845201,0.836634,0.908602,0.871134,0.859504,0.759124,0.806202,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.317400,0.323301,0.873065,0.873429,0.874619,0.873065,0.905028,0.870968,0.887671,0.833333,0.875912,0.854093,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



 Running Fold 2...


...

Map: 100%|██████████| 323/323 [00:00<00:00, 2752.79 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_29068\2799807945.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.903000,0.652839,0.625387,0.626408,0.628013,0.625387,0.679775,0.654054,0.666667,0.558621,0.586957,0.572438,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.545700,0.623167,0.752322,0.753373,0.771172,0.752322,0.847682,0.691892,0.761905,0.668605,0.833333,0.741935,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.287200,0.409636,0.835913,0.834132,0.837203,0.835913,0.826733,0.902703,0.863049,0.851240,0.746377,0.795367,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 322/322 [00:00<00:00, 7663.83 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_29068\2799807945.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.832300,0.747984,0.655280,0.603594,0.697261,0.655280,0.634058,0.945946,0.759219,0.782609,0.262774,0.393443,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.557000,0.462327,0.804348,0.805217,0.808707,0.804348,0.854651,0.794595,0.823529,0.746667,0.817518,0.780488,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.249500,0.368234,0.866460,0.866521,0.866600,0.866460,0.885870,0.881081,0.883469,0.840580,0.846715,0.843636,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [29]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_DA_fr/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [30]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)

In [31]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.874619  0.873065  0.873429  0.905028  0.870968  0.887671   
1        2        0.854211  0.854489  0.854275  0.867021  0.881081  0.873995   
2        3        0.731655  0.705882  0.681286  0.678571  0.924324  0.782609   
3        4        0.837203  0.835913  0.834132  0.826733  0.902703  0.863049   
4        5        0.866600  0.866460  0.866521  0.885870  0.881081  0.883469   
5  Average        0.832858  0.827162  0.821929  0.832645  0.892031  0.858158   

         A2                      ...   B1        B2                    C1  \
  Precision    Recall        F1  ...   F1 Precision Recall   F1 Precision   
0  0.833333  0.875912  0.854093  ...  0.0       0.0    0.0  0.0       0.0   
1  0.837037  0.818841  0.827839  ...  0.0       0.0    0.0  0.0       0.0   
2  0.802817  0.413043  0.545455  ...  0.0       0.0    0.0  0.0       0.0   
3  0.851240  0.746377  0.795367  ...  0.0       0.0    0.0  0.0       0.0   
4  0.840580  0.846715  0.843636  ...  0.0       0.0    0.0  0.0       0.0   
5  0.833001  0.740178  0.773278  ...  0.0       0.0    0.0  0.0       0.0   

                     C2              
  Recall   F1 Precision Recall   F1  
0    0.0  0.0       0.0    0.0  0.0  
1    0.0  0.0       0.0    0.0  0.0  
2    0.0  0.0       0.0    0.0  0.0  
3    0.0  0.0       0.0    0.0  0.0  
4    0.0  0.0       0.0    0.0  0.0  
5    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]